# From Accidents to Weapons: AI Incident Database analysis
Poushikan R · 2301107074 · Section B · Data Science Essentials

Is AI harm moving from accidents (AI failing) to weapons (people misusing AI on purpose)?

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
import os
import warnings
warnings.filterwarnings('ignore')
os.environ["LOKY_MAX_CPU_COUNT"]="4"   # stops a joblib warning on windows

In [ ]:
df=pd.read_excel("../data/AIID_Excel_Export-20261005.xlsx",sheet_name="Incidents",header=2)

In [ ]:
df

In [ ]:
reports=pd.read_excel("../data/AIID_Excel_Export-20261005.xlsx",sheet_name="Reports",header=1)

In [ ]:
reports.shape

In [ ]:
df.columns

# checking missing values

In [ ]:
cols=["date","year","title","description","deployer","developer","harmed","report_count",
      "Risk Domain","Risk Subdomain","Responsible Entity","Intent","Timing","Lives Lost"]
df[cols].isnull().sum()

In [ ]:
# fill % of each column
(df[cols].notnull().mean()*100).round(1)

In [ ]:
# duplicate ids
df["Incident ID"].duplicated().sum()

# cleaning the data

In [ ]:
for col in ["Risk Domain","Risk Subdomain","Responsible Entity","Intent","Timing","Data Sources"]:
    df[col]=df[col].str.strip()

df["report_count"]=pd.to_numeric(df["report_count"],errors="coerce").fillna(0).astype(int)
df["year"]=df["year"].astype(int)
df["text"]=df["title"].fillna("")+". "+df["description"].fillna("")

In [ ]:
# grouping years into eras because early years have very few incidents
# 2026 is not complete (data ends on 5 Oct)
def get_era(year):
    if year<=2017:
        return "<=2017"
    elif year<=2020:
        return "2018-20"
    elif year<=2022:
        return "2021-22"
    elif year==2026:
        return "2026*"
    else:
        return str(year)

df["era"]=df["year"].apply(get_era)
era_order=["<=2017","2018-20","2021-22","2023","2024","2025","2026*"]
df["era"].value_counts().reindex(era_order)

In [ ]:
df["Intent"].value_counts(dropna=False)

In [ ]:
df["Responsible Entity"].value_counts(dropna=False)

In [ ]:
df["Risk Domain"].value_counts(dropna=False)

In [ ]:
df["report_count"].describe()

# RQ1: how fast are AI incidents growing?

In [ ]:
yearly=df.groupby("year").size()
yearly=yearly.reindex(range(yearly.index.min(),2027),fill_value=0)
yearly.loc[2010:]

In [ ]:
# only 2010-2025 (very few incidents before 2010 and 2026 is partial)
win=yearly.loc[2010:2025]
win.describe()

In [ ]:
print("skewness:",round(stats.skew(win),2))

In [ ]:
# year on year growth %
(yearly.pct_change()*100).loc[2020:2026].round(1)

In [ ]:
x=win.index.values
y=win.values

# linear model
lin=stats.linregress(x,y)
pred_lin=lin.intercept+lin.slope*x
print("slope:",round(lin.slope,2),"incidents per year")
print("p value:",lin.pvalue)

In [ ]:
# exponential model -> log(count) = b0 + b1*(year-2010)
exp_model=sm.OLS(np.log(y),sm.add_constant(x-2010)).fit()
exp_model.summary()

In [ ]:
b0,b1=exp_model.params
pred_exp=np.exp(b0+b1*(x-2010))
growth=np.exp(b1)-1
doubling=np.log(2)/b1
print("growth per year: {:.1f}%".format(growth*100))
print("doubling time: {:.2f} years".format(doubling))

In [ ]:
# R2 on the normal scale for both models
def r2(y,pred):
    return 1-((y-pred)**2).sum()/((y-y.mean())**2).sum()

print("linear R2:",round(r2(y,pred_lin),3))
print("exponential R2:",round(r2(y,pred_exp),3))

In [ ]:
# growth before and after 2021
for start,end in [(2012,2021),(2021,2025)]:
    s=yearly.loc[start:end]
    slope=stats.linregress(s.index,np.log(s.values)).slope
    print(start,"-",end,": growth {:.1f}% per year, doubling {:.2f} years".format((np.exp(slope)-1)*100,np.log(2)/slope))

In [ ]:
plt.figure(figsize=(9,5))
plt.plot(win.index,win.values,marker="o",color="black",label="Incidents per year")
plt.plot(x,pred_exp,"--",color="grey",label="Exponential fit (+{:.0f}% / year)".format(growth*100))
plt.plot([2025,2026],[yearly[2025],yearly[2026]],":",color="black")
plt.scatter(2026,yearly[2026],facecolor="white",edgecolor="black",zorder=3)
plt.text(2026,yearly[2026]-50,"2026*\n(partial)",ha="center")
plt.axvline(2022.9,color="lightgrey")
plt.text(2022.7,400,"ChatGPT\n(Nov 2022)",ha="right")
plt.title("Fig 1: AI incidents per year (2010-2026)")
plt.xlabel("Year")
plt.ylabel("Number of incidents")
plt.legend()
plt.show()

In [ ]:
p1=(df["year"]<=2017).sum()
p2=df["year"].between(2018,2022).sum()
p3=(df["year"]>=2023).sum()
periods=pd.Series([p1,p2,p3],index=["1983-2017\n(35 years)","2018-2022\n(5 years)","2023-2026*\n(3.8 years)"])
share=periods/periods.sum()*100
share.round(1)

In [ ]:
plt.figure(figsize=(7,4.5))
plt.bar(periods.index,periods.values,color=["grey","grey","black"])
for i in range(3):
    plt.text(i,periods.values[i]+20,"{} ({:.0f}%)".format(periods.values[i],share.values[i]),ha="center")
plt.ylim(0,periods.max()*1.15)
plt.title("Fig 2: Incidents split into three periods")
plt.ylabel("Number of incidents")
plt.show()

# RQ2: is harm shifting from accidents to deliberate misuse?

In [ ]:
# only incidents labelled Intentional / Unintentional
labelled=df[df["Intent"].isin(["Intentional","Unintentional"])].copy()
labelled["is_intentional"]=(labelled["Intent"]=="Intentional").astype(int)
labelled.shape

In [ ]:
era_ct=pd.crosstab(labelled["era"],labelled["Intent"]).reindex(era_order)
era_ct

In [ ]:
era_pct=era_ct.div(era_ct.sum(axis=1),axis=0)*100
era_pct.round(1)

In [ ]:
change=era_pct.loc["2025","Intentional"]-era_pct.loc["<=2017","Intentional"]
print("change in intentional share: {:+.1f} percentage points".format(change))

In [ ]:
plt.figure(figsize=(9,5))
plt.bar(era_pct.index,era_pct["Unintentional"],color="tab:blue",label="Unintentional (accident)")
plt.bar(era_pct.index,era_pct["Intentional"],bottom=era_pct["Unintentional"],color="tab:red",label="Intentional (deliberate misuse)")
for i in range(len(era_pct)):
    u=era_pct["Unintentional"].iloc[i]
    it=era_pct["Intentional"].iloc[i]
    plt.text(i,u/2,"{:.0f}%".format(u),ha="center",color="white")
    plt.text(i,u+it/2,"{:.0f}%".format(it),ha="center",color="white")
plt.axhline(50,color="black",linewidth=0.8)
plt.title("Fig 3: Intent of harm by era")
plt.ylabel("% of labelled incidents")
plt.legend(loc="upper center",bbox_to_anchor=(0.5,-0.1),ncol=2)
plt.show()

In [ ]:
# % intentional each year (2015-2025)
yr_share=labelled.groupby("year")["is_intentional"].mean().loc[2015:2025]*100
yr_share.round(1)

In [ ]:
share_fit=stats.linregress(yr_share.index,yr_share.values)
print("slope: {:.2f} percentage points per year".format(share_fit.slope))
print("R2:",round(share_fit.rvalue**2,3),"| r:",round(share_fit.rvalue,3),"| p:",share_fit.pvalue)
print("line reaches 50% in",round((50-share_fit.intercept)/share_fit.slope,1))

In [ ]:
# logistic regression: P(intentional) vs year
logit=sm.Logit(labelled["is_intentional"],sm.add_constant(labelled["year"]-2020)).fit()
logit.summary()

In [ ]:
a,b=logit.params
print("odds ratio per year:",round(np.exp(b),3))
print("50% crossover year:",round(2020-a/b,1))

In [ ]:
# who is responsible - the AI or a human?
entity=df[df["Responsible Entity"].isin(["AI","Human"])]
ent_ct=pd.crosstab(entity["year"],entity["Responsible Entity"]).loc[2015:2025]
ent_pct=ent_ct.div(ent_ct.sum(axis=1),axis=0)*100
ent_pct.round(1)

In [ ]:
plt.figure(figsize=(9,5))
plt.plot(ent_pct.index,ent_pct["AI"],marker="o",color="tab:blue",label="AI system caused it")
plt.plot(ent_pct.index,ent_pct["Human"],marker="o",color="tab:red",label="A human caused it (using AI)")
plt.title("Fig 4: Responsible entity, share of incidents per year")
plt.xlabel("Year")
plt.ylabel("% of incidents")
plt.ylim(0,100)
plt.xticks(range(2015,2026))
plt.legend()
plt.show()

In [ ]:
# do the intent and responsible entity labels agree?
ent_lab=entity[entity["Intent"].isin(["Intentional","Unintentional"])]
(pd.crosstab(ent_lab["Intent"],ent_lab["Responsible Entity"],normalize="columns")*100).round(1)

# AI failures vs malicious misuse

In [ ]:
mal="Malicious Actors & Misuse"
saf="AI system safety, failures, and limitations"

dom_year=pd.crosstab(df["year"],df["Risk Domain"]).loc[2014:2025]
two=dom_year[[saf,mal]]
two.columns=["AI failures","Malicious misuse"]
two

In [ ]:
# linear trend for both, slope = extra incidents per year
for col in two.columns:
    fit=stats.linregress(two.index,two[col])
    print(col,": slope {:.2f} per year, R2 {:.3f}, p {:.2e}".format(fit.slope,fit.rvalue**2,fit.pvalue))

In [ ]:
plt.figure(figsize=(9,5))
plt.plot(two.index,two["AI failures"],marker="o",color="tab:blue",label="AI failures")
plt.plot(two.index,two["Malicious misuse"],marker="o",color="tab:red",label="Malicious misuse")
plt.title("Fig 5: AI failures vs malicious misuse per year")
plt.xlabel("Year")
plt.ylabel("Incidents per year")
plt.xticks(range(2014,2026))
plt.legend()
plt.show()

# change in risk domains

In [ ]:
dom_counts=df["Risk Domain"].value_counts()
dom_counts

In [ ]:
colors=[]
for d in dom_counts.index:
    if d==mal:
        colors.append("tab:red")
    elif d==saf:
        colors.append("tab:blue")
    else:
        colors.append("grey")

plt.figure(figsize=(9,4.5))
plt.barh(dom_counts.index[::-1],dom_counts.values[::-1],color=colors[::-1])
plt.title("Fig 6: MIT-labelled incidents by risk domain")
plt.xlabel("Number of incidents")
plt.show()

In [ ]:
# before 2023 vs 2023 onwards
dom=df.dropna(subset=["Risk Domain"]).copy()
dom["period"]=np.where(dom["year"]>=2023,"2023 onwards","Before 2023")
dom_pct=pd.crosstab(dom["Risk Domain"],dom["period"],normalize="columns")*100
dom_pct=dom_pct[["Before 2023","2023 onwards"]]
dom_pct["change"]=dom_pct["2023 onwards"]-dom_pct["Before 2023"]
dom_pct.sort_values("2023 onwards",ascending=False).round(1)

In [ ]:
dom_pct[["Before 2023","2023 onwards"]].sort_values("2023 onwards").plot(kind="barh",figsize=(9,5),color=["lightgrey","black"])
plt.title("Fig 7: Risk domain mix before vs after 2023")
plt.xlabel("% of incidents in that period")
plt.ylabel("")
plt.show()

In [ ]:
# same for sub domains
sub=df.dropna(subset=["Risk Subdomain"]).copy()
sub["period"]=np.where(sub["year"]>=2023,"2023 onwards","Before 2023")
sub_pct=pd.crosstab(sub["Risk Subdomain"],sub["period"],normalize="columns")*100
sub_pct.sort_values("2023 onwards",ascending=False).head(6).round(1)

# RQ3: text clustering of incident descriptions

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score, normalized_mutual_info_score

In [ ]:
extra=["ai","reportedly","allegedly","reported","according","incident","used","using","use",
       "including","claimed","said","users","user"]
stop_words=list(ENGLISH_STOP_WORDS.union(extra))

vec=TfidfVectorizer(stop_words=stop_words,ngram_range=(1,2),min_df=5,max_df=0.5,sublinear_tf=True)
X=vec.fit_transform(df["text"].str.lower())
X.shape

In [ ]:
# trying k from 2 to 12
inertia=[]
sil=[]
for k in range(2,13):
    km=KMeans(n_clusters=k,n_init=10,random_state=42).fit(X)
    inertia.append(km.inertia_)
    sil.append(silhouette_score(X,km.labels_,metric="cosine"))

In [ ]:
fig,ax=plt.subplots(1,2,figsize=(11,4))
ax[0].plot(range(2,13),inertia,marker="o")
ax[0].set_title("Elbow method")
ax[0].set_xlabel("k")
ax[0].set_ylabel("Inertia")
ax[1].plot(range(2,13),sil,marker="o")
ax[1].set_title("Silhouette score")
ax[1].set_xlabel("k")
ax[1].set_ylabel("Silhouette")
plt.suptitle("Fig 8: Choosing k (k = 7, same as the number of risk domains)")
plt.tight_layout()
plt.show()

In [ ]:
km=KMeans(n_clusters=7,n_init=10,random_state=42).fit(X)
df["cluster"]=km.labels_
terms=vec.get_feature_names_out()

for c in range(7):
    top=km.cluster_centers_[c].argsort()[::-1][:8]
    print(c,list(terms[top]))

In [ ]:
# naming the clusters after looking at the top words
def name_cluster(c):
    top=km.cluster_centers_[c].argsort()[::-1][:8]
    words=" ".join(terms[top])
    if "tesla" in words:
        return "Self-driving & Autopilot crashes"
    elif "waymo" in words or "autonomous" in words:
        return "Robots & driverless cars"
    elif "scam" in words:
        return "Deepfake scams & fraud"
    elif "school" in words:
        return "Sexual deepfake images (schools)"
    elif "facial" in words:
        return "Facial recognition & policing"
    elif "purport" in words or "trump" in words:
        return "Political deepfakes & fake media"
    elif "chatgpt" in words:
        return "Chatbots & LLM errors"
    elif "algorithm" in words:
        return "Algorithms, robots & moderation"
    else:
        return words[:30]

names={}
for c in range(7):
    names[c]=name_cluster(c)
df["cluster_name"]=df["cluster"].map(names)
df["cluster_name"].value_counts()

In [ ]:
# size and % intentional of each cluster
cl_lab=df[df["Intent"].isin(["Intentional","Unintentional"])]
cluster_pct=cl_lab.groupby("cluster_name")["Intent"].apply(lambda s:(s=="Intentional").mean()*100)
profile=pd.DataFrame({"size":df["cluster_name"].value_counts(),"pct_intentional":cluster_pct.round(0)})
profile=profile.sort_values("size")
profile

In [ ]:
colors=[]
for p in profile["pct_intentional"]:
    if p>=60:
        colors.append("tab:red")
    elif p<=40:
        colors.append("tab:blue")
    else:
        colors.append("grey")

plt.figure(figsize=(9,5))
plt.barh(profile.index,profile["size"],color=colors)
for i in range(len(profile)):
    plt.text(profile["size"].iloc[i]+5,i,"{:.0f}% intentional".format(profile["pct_intentional"].iloc[i]),va="center")
plt.xlim(0,profile["size"].max()*1.35)
plt.title("Fig 9: Text clusters (K-Means, k = 7), red = mostly intentional")
plt.xlabel("Incidents in cluster")
plt.show()

In [ ]:
# do the clusters match the expert risk domains?
dom_lab=df.dropna(subset=["Risk Domain"])
ct=pd.crosstab(dom_lab["cluster_name"],dom_lab["Risk Domain"])
purity=ct.max(axis=1).sum()/ct.values.sum()
print("purity:",round(purity,3))
print("ARI:",round(adjusted_rand_score(dom_lab["Risk Domain"],dom_lab["cluster_name"]),3))
print("NMI:",round(normalized_mutual_info_score(dom_lab["Risk Domain"],dom_lab["cluster_name"]),3))

In [ ]:
# random baseline: shuffle the cluster labels and check purity again
np.random.seed(42)
rand=[]
for i in range(200):
    shuffled=np.random.permutation(dom_lab["cluster_name"].values)
    t=pd.crosstab(shuffled,dom_lab["Risk Domain"].values)
    rand.append(t.max(axis=1).sum()/t.values.sum())
print("random purity:",round(np.mean(rand),3))

In [ ]:
# share of incidents in the "weapon" clusters (60%+ intentional) by era
weapon=profile[profile["pct_intentional"]>=60].index
df["weapon_cluster"]=df["cluster_name"].isin(weapon)
(df.groupby("era")["weapon_cluster"].mean()*100).reindex(era_order).round(1)

# RQ4: predicting intent from the text

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [ ]:
X_train,X_test,y_train,y_test=train_test_split(labelled["text"],labelled["is_intentional"],test_size=0.2,
                                               stratify=labelled["is_intentional"],random_state=42)

tfidf=TfidfVectorizer(stop_words=stop_words,ngram_range=(1,2),min_df=3,sublinear_tf=True)
X_train_vec=tfidf.fit_transform(X_train)
X_test_vec=tfidf.transform(X_test)

model=LogisticRegression(max_iter=2000,C=4)
model.fit(X_train_vec,y_train)
y_pred=model.predict(X_test_vec)

In [ ]:
print("accuracy:",round(accuracy_score(y_test,y_pred),3))
print("baseline (always guess the bigger class):",round(max(y_test.mean(),1-y_test.mean()),3))
print(classification_report(y_test,y_pred,target_names=["Unintentional","Intentional"]))

In [ ]:
cm=confusion_matrix(y_test,y_pred)
plt.figure(figsize=(5,4))
sns.heatmap(cm,annot=True,fmt="d",cmap="Blues",xticklabels=["Unintentional","Intentional"],yticklabels=["Unintentional","Intentional"])
plt.title("Fig 10: Confusion matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

In [ ]:
# words with the biggest weights
coef=pd.Series(model.coef_[0],index=tfidf.get_feature_names_out())
top_int=coef.sort_values(ascending=False).head(8)
top_unint=coef.sort_values().head(8)

fig,ax=plt.subplots(1,2,figsize=(11,4))
ax[0].barh(top_unint.index,-top_unint.values,color="tab:blue")
ax[0].set_title("Words that signal an ACCIDENT")
ax[1].barh(top_int.index,top_int.values,color="tab:red")
ax[1].set_title("Words that signal a WEAPON")
plt.suptitle("Fig 11: Strongest words in the intent classifier")
plt.tight_layout()
plt.show()

In [ ]:
# predict intent for the incidents with no label and check if the trend changes
model.fit(tfidf.fit_transform(labelled["text"]),labelled["is_intentional"])
unlabelled=df[df["Intent"].isnull()].copy()
unlabelled["is_intentional"]=model.predict(tfidf.transform(unlabelled["text"]))
print("unlabelled incidents:",len(unlabelled))

combined=pd.concat([labelled[["era","is_intentional"]],unlabelled[["era","is_intentional"]]])
compare=pd.DataFrame({"labelled only":labelled.groupby("era")["is_intentional"].mean()*100,
                      "labelled + predicted":combined.groupby("era")["is_intentional"].mean()*100})
compare.reindex(era_order).round(1)

# actors, severity and media coverage

In [ ]:
# developer/deployer/harmed can have many names separated by commas
def split_names(col):
    temp=df[["Incident ID",col,"Intent"]].dropna(subset=[col]).copy()
    temp[col]=temp[col].str.split(",")
    temp=temp.explode(col)
    temp[col]=temp[col].str.strip()
    temp=temp[~temp[col].str.contains("unknown|unidentified",case=False)]
    temp=temp[temp[col]!=""]
    return temp.drop_duplicates(["Incident ID",col])

dev=split_names("developer")
top10=dev["developer"].value_counts().head(10)
top10

In [ ]:
dev10=dev[dev["developer"].isin(top10.index)].copy()
dev10["Intent"]=dev10["Intent"].fillna("Unlabelled/Other").replace("Other","Unlabelled/Other")
dev_ct=pd.crosstab(dev10["developer"],dev10["Intent"]).reindex(columns=["Intentional","Unintentional","Unlabelled/Other"],fill_value=0)
dev_ct=dev_ct.loc[top10.index[::-1]]

dev_ct.plot(kind="barh",stacked=True,figsize=(9,5),color=["tab:red","tab:blue","lightgrey"])
plt.title("Fig 12: Top 10 developers by number of incidents")
plt.xlabel("Number of incidents")
plt.ylabel("")
plt.show()

In [ ]:
split_names("deployer")["deployer"].value_counts().head(10)

In [ ]:
split_names("harmed")["harmed"].value_counts().head(10)

In [ ]:
# news reports per incident, intentional vs unintentional
labelled.groupby("Intent")["report_count"].agg(["count","median","mean"]).round(2)

In [ ]:
# only 2023 onwards
labelled[labelled["year"]>=2023].groupby("Intent")["report_count"].agg(["count","median","mean"]).round(2)

In [ ]:
# newer incidents had less time to get news reports
print("correlation between year and report count:",round(df["year"].corr(df["report_count"]),3))

In [ ]:
# CSET severity data (only ~12% of incidents have it)
cset=df[df["Data Sources"].fillna("").str.contains("CSET")].copy()
cset["Lives Lost"]=pd.to_numeric(cset["Lives Lost"],errors="coerce")
print("CSET incidents:",len(cset))
print("total lives lost:",cset["Lives Lost"].sum())
cset[cset["Lives Lost"]>0].sort_values("Lives Lost",ascending=False)[["year","title","Lives Lost","Intent"]].head(8)

In [ ]:
# news articles per year
reports["Date Published"]=pd.to_datetime(reports["Date Published"],errors="coerce")
reports["year"]=reports["Date Published"].dt.year
articles=reports[reports["year"].between(2013,2026)]["year"].astype(int).value_counts().sort_index()
articles

In [ ]:
plt.figure(figsize=(9,5))
plt.plot(articles.loc[:2025].index,articles.loc[:2025].values,marker="o",color="black")
plt.plot([2025,2026],[articles[2025],articles[2026]],":",color="black")
plt.scatter(2026,articles[2026],facecolor="white",edgecolor="black",zorder=3)
plt.title("Fig 13: News articles about AI incidents per year (2026* partial)")
plt.xlabel("Year")
plt.ylabel("Articles published")
plt.xticks(range(2013,2027),rotation=45)
plt.show()

# conclusion
* incidents are growing exponentially, and faster after 2021
* intentional harm went from about 1 in 4 incidents to about 2 in 3, and misuse overtook AI failures in 2023
* the text alone separates into "weapon" clusters (deepfakes, scams) and "accident" clusters (self-driving)
* a simple text classifier predicts intent well above the baseline, and filling the missing labels does not change the trend

**Hypothesis "AI harm is shifting from accidental failure to deliberate misuse": supported.**